In [ ]:
%matplotlib inline

In [ ]:
# Import libraries here

import numpy as np
import pandas as pd
import os
import sys
import matplotlib.pyplot as plt
import scipy.optimize as sco

In [ ]:
current_dir = os.path.abspath('')
project_root = os.path.abspath(os.path.join(current_dir, '../../'))

if project_root not in sys.path:
    sys.path.append(project_root)

os.chdir(project_root)

print(f"Working directory set to: {os.getcwd()}")

In [ ]:
# Import modules here

import importlib
from src.plotting_utils import plotting_utils as plot_utils
from src.data_pipeline_utils import data_fetching_handling as data_pipe
from src.mathematical_calculations_utils import markowitz_calc_utils as m_calc
importlib.reload(plot_utils)
importlib.reload(data_pipe)

# Efficient Frontier – Mathematical Approach

In the previous notebook, the efficient frontier was approximated using a Monte Carlo simulation or we tried different number of attempts to guess the optimal portfolio. Simulation or guessing is not the right mathematical approach as even 1 ot 2 million runs are not enough to satisfy the law or large numbers. Even for a portfolio of 10 stocks, we may need tens of millions of guesses to reach the best result. 

In this notebook, we derive the efficient frontier analytically using the Markowitz mean–variance framework and I will try to portray the full concept mathematically as it was developed originally in 1952.

The goal is to show that the efficient frontier can be expressed as a quadratic function of expected portfolio return. We are going to reach an important conclusion about constraints, get to the point where mathematics will become extremely complex, which will guide us to the next notebook, where we will seek algorithmic implementation in the solution of the very high level of complexity.

# Modern portfolio theory (MPT) is the hallmark of contemporary understanding of investment strategies. It first introduced the notion that: #

1. An investor can select an optimal portfolio, which maximizes return for a given level of risk
2. An investor can find a portfolio, which minimizes risk given a targeted return

This happens through diversification or owning many different risk-bearing assets instead of a few or a single one. For example owning shares only in Nvidia (NVDA) could make a great return as we see lately in the period between 2024 and 2025, but on the hand owning only Bitcoin (BTC) can brung great losses if we focus on the period between 2025 and March 2026. 

What is a risk asset exactly? The notion of risk is characterised by the variance of its daily price around the mean return for a given period. The higher the variance, the riskier is the asset. 

[The founding study of MPT was published by scientist Harry Markowitz in 1952 and is simplistically described in my favorite online finance reference guide.](https://www.investopedia.com/terms/m/modernportfoliotheory.asp)

What did actually Markowitz solve? Mathematically and conceptually he introduced just three fundamental concepts as follows:

#### 1. Portfolio variance: ####
$$ \sigma_p^2 = \sum_{i=1}^{n} w_i^2 \sigma_i^2 + \sum_{i \neq j} w_i w_j \operatorname{Cov}_{ij} $$

or in Matrix form if we take weights of assets in the portfolio and their individual expected returns as vectors:

$$ \sigma_p^2 = \mathbf{w}^\top \Sigma \mathbf{w} $$

In other words, portfolio risk includes:
- Variance of each asset
- Covariance between assets

Or this is the mathematics behind diversification. 

#### 2. The expected return of a portfolio ####
$$ \mu_p = \sum_{i=1}^{n} w_i \mu_i $$

#### 3. And the optimization solution is to minimize the portfolio variance: ####
$$ \min_{\mathbf{w}} \quad \mathbf{w}^\top \Sigma \mathbf{w} $$

if we assume that $ \mathbf{w}^\top \boldsymbol{\mu} = \mu_p $ is true and we are fully invested and without leverage $ \mathbf{1}^\top \mathbf{w} = 1 $

So Markowitz has applied a rule in Mathematics called the [generic Lagrange rule](https://alexkritchevsky.com/2024/06/10/lagrange-multipliers.html) in order to derive the below function seeking optimal solution for the portfolio variance given the two constraints:

$$ \mathcal{L}(\mathbf{w},\lambda,\gamma) = \mathbf{w}^\top \Sigma \mathbf{w} - \lambda (\mathbf{w}^\top \boldsymbol{\mu} - \mu_p) - \gamma (\mathbf{1}^\top \mathbf{w} - 1) $$

---

### Step 1: First we take the first derivative: ###

$$ \nabla_{\mathbf{w}} \mathcal{L} = 2\Sigma \mathbf{w} - \lambda \boldsymbol{\mu} - \gamma \mathbf{1} = 0 $$

---

### Step 2: Then we solve for weights in the portfolio as follows: ###

$$ 2\Sigma \mathbf{w} = \lambda \boldsymbol{\mu} + \gamma \mathbf{1} $$

---

### Step 3: Multiply both sides by the inverse covariance matrix ###

Multiply on the left by $\Sigma^{-1}$:

$$ \Sigma^{-1}(2\Sigma \mathbf{w}) = \Sigma^{-1}(\lambda \boldsymbol{\mu} + \gamma \mathbf{1}) $$

Since:

$$ \Sigma^{-1}\Sigma = I $$

(where $I$ is the identity matrix),

we obtain:

$$ 2\mathbf{w} = \lambda \Sigma^{-1}\boldsymbol{\mu} + \gamma \Sigma^{-1}\mathbf{1} $$

---

### Step 4: Divide both sides by 2

$$
\mathbf{w}
=
\frac{\lambda}{2}\Sigma^{-1}\boldsymbol{\mu}
+
\frac{\gamma}{2}\Sigma^{-1}\mathbf{1}
$$

This shows that the optimal weights are a linear combination of:

- $\Sigma^{-1}\boldsymbol{\mu}$ (risk-adjusted return direction)
- $\Sigma^{-1}\mathbf{1}$ (risk-adjusted budget direction)

---

## Step 5: Apply the Constraints

We now use the two constraints:

1. Target return:

$$
\mathbf{w}^\top \boldsymbol{\mu} = \mu_p
$$

2. Full investment:

$$
\mathbf{1}^\top \mathbf{w} = 1
$$

Substituting the expression for $\mathbf{w}$ into both constraints leads to repeated matrix expressions.

To simplify notation, we define:

$$
A = \mathbf{1}^\top \Sigma^{-1} \mathbf{1}
$$

$$
B = \mathbf{1}^\top \Sigma^{-1} \boldsymbol{\mu}
$$

$$
C = \boldsymbol{\mu}^\top \Sigma^{-1} \boldsymbol{\mu}
$$

We also define:

$$
D = AC - B^2
$$

---

## Step 6: Solving for the Multipliers

After substituting into the two constraints and solving the resulting 2×2 linear system, we obtain:

$$
\lambda = \frac{2(A\mu_p - B)}{D}
$$

$$
\gamma = \frac{2(C - B\mu_p)}{D}
$$

---

## Step 7: Efficient Frontier Equation

Substituting the optimal weights back into the portfolio variance:

$$
\sigma_p^2 = \mathbf{w}^\top \Sigma \mathbf{w}
$$

yields the closed-form equation of the efficient frontier:

$$
\sigma^2(\mu_p)
=
\frac{A\mu_p^2 - 2B\mu_p + C}{D}
$$

This is a quadratic function in $\mu_p$, which explains why the efficient frontier is a parabola in mean-variance space.

To start, we take again our stock symbol tickers. Again let's keep them up to 10 for now. In fact, for the sake of experiments, let us keep them the same as in notebook 1_1, where we practiced portfolio simulation.

In [ ]:
tickers = ['AAPL', 'NVDA', 'MSFT', 'JNJ', 'BAC', 'VZ', 'WMT', 'UPS', 'PFE', 'JPM']

Let's build the now familiar Pandas dataframe returns_df

In [ ]:
returns_df =  data_pipe.build_returns_df(tickers)

### Let's run calculations through theory as outlined ###

Some of the steps below look familiar to what we did with the monte carlo simulation. We would take the mean returns of the stocks, we will define the covariance matrix. Here I will show that we can construct a portfolio with equal weights of the stocks and through matrix multiplication we can define the expected portfolio return. While in the monte carlo we just randomly selected stock weights and calculated many different portfolios with their variance and return combinations, here we will apply the mathematical principles as outlined in the steps above.

In [ ]:
trading_days = 252
mean_returns = returns_df.mean() * trading_days
cov_matrix = returns_df.cov() * trading_days

In [ ]:
number_of_stocks = len(tickers)
weights = np.array([1 / number_of_stocks] * number_of_stocks)

In [ ]:
# Calculate the expected portfolio return
expected_portfolio_return = mean_returns @ weights
print(expected_portfolio_return)

In [ ]:
# Calculate the inverse of the covariance matrix (Σ⁻¹)
inv_cov_matrix = np.linalg.inv(cov_matrix)
print(inv_cov_matrix)

In [ ]:
# Define the vector of ones (1)
ones_vector = np.ones(number_of_stocks)
print(ones_vector)

In [ ]:
# Compute A = 1^T Σ⁻¹ 1
A = ones_vector.T @ inv_cov_matrix @ ones_vector

# Compute B = 1^T Σ⁻¹ μ
B = ones_vector.T @ inv_cov_matrix @ mean_returns

# Compute C = μ^T Σ⁻¹ μ
C = mean_returns.T @ inv_cov_matrix @ mean_returns

# Compute D = AC - B^2
D = A * C - B**2

print(f"A = {A:.4f}")
print(f"B = {B:.4f}")
print(f"C = {C:.4f}")
print(f"D = {D:.4f}")

In [ ]:
# Solve the optimization for a target portfolio return
target_return_example = 0.20
optimal_weights = m_calc.calculate_optimal_weights(
    target_return_example, inv_cov_matrix, mean_returns, ones_vector, A, B, C, D
)

optimal_portfolio_volatility = np.sqrt(np.dot(optimal_weights.T, np.dot(cov_matrix, optimal_weights)))

print(f"Optimal weights for a {target_return_example*100}% return:")
print(pd.Series(optimal_weights, index=tickers))

In [ ]:
# Generate a range of target returns (e.g., from 0% to 40%)
target_returns_range = np.linspace(0.0, 0.40, 100)

# Calculate exact variances and volatilities (standard deviations)
analytical_variances = m_calc.calculate_efficient_frontier_variance(target_returns_range, A, B, C, D)
analytical_volatilities = np.sqrt(analytical_variances)

At this point we have arrived at the solution of the original Harry Markowitz foundation of MPT! It gets very interesting to plot the results and observe:

In [ ]:
plot = plot_utils.plot_taylor_expansion(analytical_volatilities,
                                        target_returns_range, 
                                        optimal_portfolio_volatility,
                                        target_return_example,
                                        'Target Returns for Given Risk',
                                        'Analytical Volatilities',
                                        'Target Returns'
                                       )
plt.show()

### What can be observed? ###

The shape of the graph resembles strongly the output of the Monte Carlo simulation we have already performed! Applying the mathematical concepts allows us to arrive at the true efficient frontier. 

However, let's calculate the weights of all portfolios lying on the efficient frontier curve we derived above and save them in a numpy array. We are going to find out that most likely all portfolios will have negative weights in them.

This actually is a problem, becuase in almost all the cases in reality, this theory is applied for the creation of long-only portfolios, where short-selling (selling assets, which we do not possess) is not allowed.

In [ ]:
frontier_weights = m_calc.calculate_optimal_weights_for_range(target_returns_range, inv_cov_matrix, mean_returns, ones_vector, A, B, C, D)
print(frontier_weights)

### Assumptions of the Markowitz Framework

The Markowitz mean–variance model relies on several simplifying assumptions:

- Asset returns are sufficiently described by their mean and variance.
- Investors are risk-averse and prefer higher expected return for a given level of risk.
- Covariance between assets remains stable over time.
- Markets are frictionless (no taxes, transaction costs, or liquidity constraints).

In practice, financial return distributions often exhibit (negative) skewness, heavy tails (positive kurtosis), and also time-varying correlations, which can limit the predictive power of mean–variance optimization.

### Conclusion ###

The theory is great, the mathematics behind is fascinating. It has become the norm in investment practice in the whole world. 

However, as we observed just a step above - there is a negative weight in (almost) every portfolio-weights combination forming the shape of the efficient frontier given the selected stock tickers for the example. What does a negative weight mean? This is short-selling, or selling something you do not own. But portfolios with short selling are rarely utilized by the mass investor. 

We need to tighten the constraint -> not only the weight sum up to one, but every weight is greater than or equal to zero, i.e. we do not allow to short sell stocks in order to construct our portfolio.

So this immediately stops being a quadratic function of portfolio variance with two linear constraints. The math becomes extremely complex, and most of us see the challenge to even understand the seven steps above.

So we move to our next notebook 1_3 and put scientific and financial python libraries to work in order to order to define the function and seek its global minimum in order to identify the optimal portfolio and to plot the lowest-variance portfolios for each targeted return - the Markowitz efficient frontier. This time with long-only portfolios.